# 99 — Target-environment smoke tests and inventory

Run these tiny tests before whole-slide processing. This bundle has local control/unit tests; full SpatialData, S3 and GPU execution must be verified on the target machine.
Tests use the configured local scratch, not an implicit /tmp directory. Optional scientific tests report skipped packages; skipped is not passed.
No real biological sample is used. `VHD_TEST_SCVI=1` separately enables the small CPU scVI test in the scVI interpreter.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vhd" / "control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the extracted pipeline folder (or its notebooks folder).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from vhd.control.manifest import load_project, check_policy, save_plan, sample_layout
MANIFEST = Path(os.environ.get("VHD_MANIFEST", ROOT / "config" / "samples.csv"))
SETTINGS = Path(os.environ.get("VHD_SETTINGS", ROOT / "config" / "settings.json"))
if not MANIFEST.exists() or not SETTINGS.exists():
    raise FileNotFoundError("Copy a supplied samples.*.csv to config/samples.csv and settings.g5_24xlarge.example.json to config/settings.json; edit paths and policy first.")
PROJECT = load_project(MANIFEST, SETTINGS)
check_policy(PROJECT)
# Default is a dry run. Set True here only after reviewing the printed plan.
EXECUTE = os.environ.get("VHD_EXECUTE", "0") == "1"
# Optional pilot selection, e.g. ["StudyLegacy__Sample01"]. None selects all applicable rows.
SAMPLE_KEYS = None


## Inventory each configured environment in a separate process

In [ ]:
from vhd.compute.launch import worker_env, run_logged
scratch = Path(PROJECT["settings"]["integration"]["tmp_dir"]) / "smoke" / PROJECT["settings"]["run_id"]
for key in ("python_spatial", "python_stardist", "python_scvi", "python_rapids"):
    command = [PROJECT["settings"]["execution"][key], "-m", "vhd.compute.probe"]
    print(command)
    if EXECUTE and not Path(command[0]).is_file():
        print("Skipped missing interpreter; this environment has NOT been tested:", command[0])
        continue
    if EXECUTE:
        log = scratch / (key + ".log")
        run_logged(command, worker_env(scratch/key, [], 2), log)
        print(log.read_text())

## Real AnnData and SpatialData roundtrip tests in the configured spatial environment

In [ ]:
command = [PROJECT["settings"]["execution"]["python_spatial"], "-m", "pytest", "-q",
           str(ROOT / "tests"), "--basetemp", str(scratch / "pytest_tmp")]
print(command)
if EXECUTE:
    run_logged(command, worker_env(scratch/"pytest_runtime", [], 2), scratch/"pytest.log")
    print((scratch/"pytest.log").read_text())